<a href="https://colab.research.google.com/github/LizethArista/Programacion/blob/main/Ayudant%C3%ADas/Ejemplo_Diccionarios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## RECORDEMOS...

Los diccionarios usan tablas hash, por lo que las claves deben ser inmutables (str, int, tuple puro)

Las tuplas permiten crear claves compuestas, ideales para finanzas: ("2024-Q1", "INGRESOS"), (id_cliente, fecha), etc

Si una tupla contiene un objeto mutable (lista, dict), pierde su hashabilidad

Solución: tuple() o frozenset()

En contabilidad: las reglas son fijas (tasas, periodos), por lo que podemos usar tuplas. En caso de que los movimientos cambien, se debe usar un diccionario


In [8]:
saldos_periodicos = {
    ("2024-Q1", "ingresos"): 15000.0,
    ("2024-Q1", "gastos"): 8500.0,
    ("2024-Q2", "ingresos"): 18200.0
}
print("Saldo Q1 Ingresos:", saldos_periodicos[("2024-Q1", "ingresos")])

Saldo Q1 Ingresos: 15000.0


tasas_impuesto: tupla fija (iva, retencion)

movimientos: diccionario mutable con claves compuestas (fecha, cuenta, tipo)

Métodos: registrar(), consultar_saldo(), calcular_impuestos(), exportar_estado() (devuelve tupla inmutable para reportes)


In [1]:
class GestorContable:
    def __init__(self, tasas_impuesto: tuple):
        if len(tasas_impuesto) != 2:
            raise ValueError("Las tasas deben ser una tupla: (iva, retencion)")
        self.tasas_impuesto = tasas_impuesto  # Inmutable por diseño
        self.movimientos = {}                 # Mutable: {(fecha, cuenta, tipo): monto}

    def registrar(self, fecha: str, cuenta: str, tipo: str, monto: float):
        clave = (fecha, cuenta, tipo)
        # Si ya existe la clave, se actualiza (mutabilidad del diccionario)
        self.movimientos[clave] = monto
        print(f"Registrado: ${monto:,.2f} en '{cuenta}' ({tipo}) para {fecha}")

    def consultar_saldo(self, cuenta: str, tipo: str = "todos") -> float:
        total = 0.0
        for (fecha, c, t), monto in self.movimientos.items():
            if c == cuenta and (tipo == "todos" or t == tipo):
                total += monto
        return total

    def calcular_impuestos(self, cuenta_ingresos: str = "ventas") -> dict:
        ingresos = self.consultar_saldo(cuenta_ingresos, "credito")
        iva, ret = self.tasas_impuesto
        return {
            "ingresos_brutos": ingresos,
            "iva_a_pagar": ingresos * iva,
            "retencion": ingresos * ret
        }

    def exportar_estado(self) -> tuple:
        # Devuelve una tupla inmutable para reportes
        ventas = self.consultar_saldo("ventas", "credito")
        gastos = self.consultar_saldo("gastos_operativos", "debito")
        utilidad = ventas - gastos
        return ("Estado Financiero", ventas, gastos, utilidad, self.tasas_impuesto)

In [9]:
# Inicializamos
contabilidad = GestorContable(tasas_impuesto=(0.19, 0.10)) # IVA 19%, Retención 10%

# Registramos movimientos
contabilidad.registrar("2024-05-01", "ventas", "credito", 10000.0)
contabilidad.registrar("2024-05-02", "gastos_operativos", "debito", 3500.0)
contabilidad.registrar("2024-05-05", "ventas", "credito", 5000.0)


Registrado: $10,000.00 en 'ventas' (credito) para 2024-05-01
Registrado: $3,500.00 en 'gastos_operativos' (debito) para 2024-05-02
Registrado: $5,000.00 en 'ventas' (credito) para 2024-05-05


In [10]:
# Consultas y cálculos
print(f"\n Total Ventas: ${contabilidad.consultar_saldo('ventas', 'credito'):,.2f}")
print(f" Total Gastos: ${contabilidad.consultar_saldo('gastos_operativos', 'debito'):,.2f}")



 Total Ventas: $15,000.00
 Total Gastos: $3,500.00


In [11]:
# Cálculo de impuestos
impuestos = contabilidad.calcular_impuestos()
print(f"\n Impuestos calculados:")
for k, v in impuestos.items():
    print(f"   {k}: ${v:,.2f}")


 Impuestos calculados:
   ingresos_brutos: $15,000.00
   iva_a_pagar: $2,850.00
   retencion: $1,500.00


In [ ]:
# EJERCICIO... (Resuelve el forms)
# Extiende la clase para manejar múltiples divisas. Usa una clave compuesta de 4 elementos:
# (fecha, cuenta, tipo, moneda). Mantén la inmutabilidad y agrega un método consultar_por_moneda(moneda: str)

class GestorContableMultidivisa(GestorContable):
    def __init__(self, tasas_impuesto: tuple):
        super().__init__(tasas_impuesto)
        # self.movimientos heredado, pero ahora usaremos 4 elementos como clave

    def registrar(self, fecha: str, cuenta: str, tipo: str, monto: float, moneda: str = "USD"):
        # Crea una tupla de 4 elementos y regístrala en self.movimientos
        pass

    def consultar_por_moneda(self, moneda: str) -> dict:
        # Devuelve un dict {cuenta: total} filtrando por moneda
        pass

# Prueba esperada:
# gestor = GestorContableMultidivisa((0.19, 0.10))
# gestor.registrar("2024-05-10", "banco_exterior", "credito", 2000.0, "USD")
# print(gestor.consultar_por_moneda("USD"))